# 🎯 3. Query Transformations

Το πιο **υποτιμημένο** πρόβλημα του naive RAG: ο χρήστης γράφει μια ερώτηση με **διαφορετικές λέξεις** από αυτές που υπάρχουν στα docs. Παρόλο που τα embeddings είναι «semantic», η αναζήτηση συχνά **αστοχεί** όταν το vocabulary δεν ταιριάζει.

**Παράδειγμα:**
* Doc: «Το Pulsar tier προσφέρει 99.999% uptime guarantee»
* Ερώτηση: «Ποιο SLA έχω αν χρειαστώ mission-critical reliability;»

Σημασιολογικά συνδέονται, αλλά οι λέξεις-κλειδιά (uptime ↔ SLA, Pulsar ↔ mission-critical) δεν επικαλύπτονται. Συχνά ο top-k retriever αστοχεί.

Οι **query transformations** αλλάζουν την ερώτηση **πριν** το retrieval για να αυξήσουν τις πιθανότητες σωστού match. Θα δούμε:

1. **Multi-Query** — πολλές διατυπώσεις
2. **RAG-Fusion** — multi-query + Reciprocal Rank Fusion
3. **Decomposition** — διάσπαση σε υπο-ερωτήσεις
4. **Step-Back** — από συγκεκριμένο σε γενικό
5. **HyDE** — υποθετικό document πριν την αναζήτηση

<img src="images/query_transformations_overview.png" width="80%" style="border-radius:10px;margin:12px 0;"/>

***Εικ. 3.1** — Οι 5 τεχνικές Query Transformation. Κάθε μία αντιμετωπίζει διαφορετική αιτία αποτυχίας του naive retrieval.*

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(__file__).parent / ".env" if "__file__" in dir() else Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

# Αν δεν βρεθεί το key (πχ σε Colab), ζητάμε manually
# if not os.environ.get("OPENAI_API_KEY"):
#     import getpass
#     os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

LLM_MODEL   = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

## 3.1 Setup — knowledge base για όλες τις τεχνικές

In [2]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

embedder = OpenAIEmbeddings(model=EMBED_MODEL)
llm      = ChatOpenAI(model=LLM_MODEL, temperature=0)

# Datanous.ai knowledge base — defined once, reused throughout this notebook
datanous_kb = [
    Document(page_content="Datanous Insight is a RAG-powered knowledge management platform that indexes enterprise documents and answers queries with source citations.",
             metadata={"source": "datanous_products.txt", "product": "Insight"}),
    Document(page_content="Datanous Insight Starter: 50 USD per month, up to 10,000 document pages, single-tenant isolation, community support.",
             metadata={"source": "datanous_products.txt", "product": "Insight", "tier": "Starter"}),
    Document(page_content="Datanous Insight Professional: 350 USD per month, up to 100,000 pages, multi-tenant row-level access control, email support.",
             metadata={"source": "datanous_products.txt", "product": "Insight", "tier": "Professional"}),
    Document(page_content="Datanous Insight Enterprise: unlimited pages, dedicated vector store, 99.9 percent uptime SLA, custom pricing, on-premises deployment available.",
             metadata={"source": "datanous_products.txt", "product": "Insight", "tier": "Enterprise"}),
    Document(page_content="Datanous Search is a hybrid retrieval API combining BM25 and dense semantic embeddings via Reciprocal Rank Fusion. A cross-encoder reranker selects the final top-k results. Latency under 200 ms at the 95th percentile over one million chunks.",
             metadata={"source": "datanous_products.txt", "product": "Search"}),
    Document(page_content="Datanous Guard enforces four checks on every request: (1) factual grounding, (2) prompt injection detection, (3) PII redaction of names, emails, and IDs, (4) tenant isolation via tenant_id filtering.",
             metadata={"source": "datanous_products.txt", "product": "Guard"}),
    Document(page_content="Datanous Embed supports text-embedding-3-small (1536 dims) and text-embedding-3-large (3072 dims). Batch up to 2048 texts. Redis cache hit rate exceeds 60 percent on enterprise corpora.",
             metadata={"source": "datanous_products.txt", "product": "Embed"}),
    Document(page_content="All Datanous.ai deployments encrypt data in transit with TLS 1.3 and at rest with AES-256. The company is GDPR-compliant and provides a Data Processing Agreement for Professional and Enterprise clients.",
             metadata={"source": "datanous_faq.txt", "topic": "privacy"}),
    Document(page_content="Datanous.ai serves clients in banking, legal, healthcare, and media across Europe. A law firm deployment reduced search time from 2.5 hours to 8 minutes with 91 percent faithfulness.",
             metadata={"source": "datanous_case_studies.txt", "section": "results"}),
    Document(page_content="A retail bank deployed Datanous Insight Enterprise over 12,000 regulatory pages, cutting query resolution from 4 days to under 30 seconds.",
             metadata={"source": "datanous_case_studies.txt", "industry": "banking"}),
    Document(page_content="Datanous.ai evaluates every deployment on faithfulness, answer relevancy, context precision, and context recall using an LLM-as-judge pipeline.",
             metadata={"source": "datanous_blog_rag.txt", "section": "evaluation"}),
    Document(page_content="Datanous.ai is a certified partner of OpenAI, Anthropic, Pinecone, and Weaviate, and builds on LangChain, LangGraph, and LangSmith.",
             metadata={"source": "datanous_overview.txt", "section": "partners"}),
]

vectorstore = Chroma.from_documents(datanous_kb, embedding=embedder)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

print(f"Knowledge base ready: {vectorstore._collection.count()} documents indexed")


Knowledge base ready: 12 documents indexed


## 3.2 Multi-Query — πολλές διατυπώσεις της ίδιας ερώτησης

**Ιδέα:** Παράγουμε με ένα LLM πολλές παραλλαγές της ερώτησης (πχ 4-5). Κάνουμε retrieval **για κάθε μία**. Παίρνουμε **union** των αποτελεσμάτων.

Αυτό αυξάνει την πιθανότητα να «πιάσουμε» το σωστό vocabulary.

<img src="images/multi-query.png" width="80%" style="border-radius:10px;margin:12px 0;"/>


In [3]:
from langchain_core.prompts import ChatPromptTemplate

MULTI_QUERY_PROMPT = ChatPromptTemplate.from_template(
    """You are an AI assistant helping to improve document retrieval for Datanous.ai.
Given the user question below, generate {n} different reformulations of it.
Each reformulation should approach the topic from a different angle to maximise recall.
Return only the questions, one per line, with no numbering or extra text.

Original question: {question}"""
)

def parse_queries(text: str) -> list[str]:
    return [q.strip() for q in text.strip().split("\n") if q.strip()]

multi_query_gen = MULTI_QUERY_PROMPT | llm | StrOutputParser() | parse_queries

# Test: generate 3 reformulations of a Datanous.ai support question
test_q = "What security features does Datanous.ai offer?"
queries = multi_query_gen.invoke({"question": test_q, "n": 3})
print(f"Original: {test_q}\n")
print("Reformulations:")
for i, q in enumerate(queries, 1):
    print(f"  {i}. {q}")


Original: What security features does Datanous.ai offer?

Reformulations:
  1. What are the security measures implemented by Datanous.ai?
  2. Can you list the protective features available in Datanous.ai?
  3. How does Datanous.ai ensure data security for its users?


Τώρα κάνουμε retrieval για κάθε variation και παίρνουμε union.

In [4]:
# Generate multiple query reformulations and retrieve for each
# The union of results gives broader recall than a single query

def multi_query_retrieval(question: str, n: int = 3) -> list:
    """Generate n query variants, retrieve for each, fuse with RRF."""
    queries     = multi_query_gen.invoke({"question": question, "n": n})
    all_results = [retriever.invoke(q) for q in queries]
    fused       = reciprocal_rank_fusion(all_results)
    return fused

# We define reciprocal_rank_fusion here for use above — full version in section 3.3
def reciprocal_rank_fusion(results_lists: list, k: int = 60) -> list:
    scores, doc_map = {}, {}
    for results in results_lists:
        for rank, doc in enumerate(results):
            key = doc.page_content
            doc_map[key] = doc
            scores[key] = scores.get(key, 0) + 1 / (k + rank + 1)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [(doc_map[c], s) for c, s in ranked]

# Test multi-query retrieval with a Datanous.ai question
test_q  = "What security features does Datanous.ai offer?"
queries = multi_query_gen.invoke({"question": test_q, "n": 3})
print(f"Query: {test_q}")
print("Reformulations:")
for i, q in enumerate(queries, 1):
    print(f"  {i}. {q}")

results = multi_query_retrieval(test_q)
print(f"\nFused results ({len(results)} unique docs):")
for doc, score in results[:3]:
    print(f"  [{score:.4f}] {doc.page_content[:80]}")


Query: What security features does Datanous.ai offer?
Reformulations:
  1. What are the security measures implemented by Datanous.ai?
  2. Can you list the protective features available in Datanous.ai?
  3. How does Datanous.ai ensure data security for its users?

Fused results (4 unique docs):
  [0.0492] All Datanous.ai deployments encrypt data in transit with TLS 1.3 and at rest wit
  [0.0476] Datanous.ai serves clients in banking, legal, healthcare, and media across Europ
  [0.0323] Datanous Guard enforces four checks on every request: (1) factual grounding, (2)


**Trade-off:** Πολλαπλό retrieval = περισσότερες κλήσεις. Σε production μπορείς να κάνεις **batch** τα queries μαζί.

## 3.3 RAG-Fusion — multi-query + smart ranking

Πρόβλημα του απλού multi-query: παίρνουμε union αλλά **χάνουμε την πληροφορία ranking**. Κάποια docs εμφανίζονται **πολλές φορές** σε διαφορετικές queries — αυτό είναι σήμα ότι είναι όντως σχετικά.

Το **Reciprocal Rank Fusion (RRF)** το αξιοποιεί:

$$\text{RRF}(d) = \sum_{q \in Q} \frac{1}{k + \text{rank}_q(d)}$$

Όπου:
* `Q` = όλες οι παραλλαγές της ερώτησης
* `rank_q(d)` = η θέση του doc `d` στα αποτελέσματα της query `q`
* `k` = constant (συνήθως 60) που αμβλύνει την επίδραση του ranking στις πρώτες θέσεις

**Ιδιότητες:**
* Όσο πιο ψηλά εμφανίζεται ένα doc → μεγαλύτερο score
* Όσο πιο πολλές queries το επιστρέφουν → μεγαλύτερο score
* Δεν χρειάζεται κανονικοποίηση scores από διαφορετικά retrievers

In [5]:
def reciprocal_rank_fusion(results_list: list[list], k: int = 60) -> list[tuple]:
    """Merge multiple ranked lists into one using Reciprocal Rank Fusion."""
    scores: dict = {}
    for results in results_list:
        for rank, doc in enumerate(results):
            key = doc.page_content
            scores[key] = scores.get(key, 0) + 1 / (k + rank + 1)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    # Return (doc, score) tuples matching the best document object
    doc_map = {d.page_content: d for rl in results_list for d in rl}
    return [(doc_map[content], score) for content, score in ranked]

def multi_query_retrieval(question: str, n: int = 3) -> list:
    """Generate n query variants, retrieve for each, fuse with RRF."""
    queries = multi_query_gen.invoke({"question": question, "n": n})
    all_results = [retriever.invoke(q) for q in queries]
    fused = reciprocal_rank_fusion(all_results)
    return fused

question = "What security and compliance features does Datanous.ai provide?"
results  = multi_query_retrieval(question)

print(f"Query: {question}\n")
print("Fused results (RRF ranked):")
for doc, score in results[:4]:
    print(f"  [{score:.4f}] {doc.page_content[:100]}")


Query: What security and compliance features does Datanous.ai provide?

Fused results (RRF ranked):
  [0.0492] All Datanous.ai deployments encrypt data in transit with TLS 1.3 and at rest with AES-256. The compa
  [0.0323] Datanous.ai is a certified partner of OpenAI, Anthropic, Pinecone, and Weaviate, and builds on LangC
  [0.0320] Datanous.ai serves clients in banking, legal, healthcare, and media across Europe. A law firm deploy
  [0.0159] Datanous.ai evaluates every deployment on faithfulness, answer relevancy, context precision, and con


Παρατήρησε ότι τα top documents τώρα είναι αυτά που εμφανίζονταν συστηματικά **σε όλες** τις παραλλαγές.

### Σύνδεση RAG-Fusion με LLM

In [6]:
from langchain_core.prompts import ChatPromptTemplate

FINAL_PROMPT = ChatPromptTemplate.from_template(
    """You are a technical assistant for Datanous.ai.
Answer the question using ONLY the context below.
If the answer is not in the context, say: "I do not have that information."

Context:
{context}

Question: {question}

Answer:"""
)

def fusion_retrieval(question: str) -> list[tuple]:
    """Full RAG-Fusion: multi-query -> RRF -> top documents."""
    return multi_query_retrieval(question)

rag_fusion_chain = (
    {
        "context": lambda x: format_docs(
            [doc for doc, _ in fusion_retrieval(x["question"])[:3]]
        ),
        "question": lambda x: x["question"],
    }
    | FINAL_PROMPT | llm | StrOutputParser()
)

answer = rag_fusion_chain.invoke({
    "question": "Which Datanous.ai plan offers the highest reliability for production?"
})
print("Q: Which Datanous.ai plan offers the highest reliability for production?")
print("A:", answer)


Q: Which Datanous.ai plan offers the highest reliability for production?
A: Datanous Insight Enterprise offers the highest reliability for production with a 99.9 percent uptime SLA.


<img src="images/decomposition_strategies.png" height="700px"  style="border-radius:10px;margin:12px 0;"/>

***Εικ. 3.2** — Decomposition: δύο στρατηγικές. Αριστερά: Individual (parallel). Δεξιά: Recursive (sequential, cumulative context).*

## 3.4 Decomposition — σπάμε σύνθετες ερωτήσεις

Όταν ο χρήστης κάνει μια **σύνθετη** ερώτηση που απαιτεί **πολλά κομμάτια πληροφορίας**, ένα μόνο retrieval συχνά δεν αρκεί.

**Παράδειγμα:** *«Θέλω uptime > 99.9%, encryption και αυτόματο failover. Ποιο tier ταιριάζει;»* — αυτό είναι **3 ξεχωριστές** ερωτήσεις σε μία.

Η **decomposition** σπάει την ερώτηση σε υπο-ερωτήσεις και τις απαντά μία προς μία, στη συνέχεια συνθέτει.

In [7]:
DECOMPOSE_PROMPT = ChatPromptTemplate.from_template(
    """You are an expert at query decomposition for a Datanous.ai support assistant.

Given a complex question, break it into at most 3 simpler, independent sub-questions
that together cover the original. Return only the sub-questions, one per line.

Complex question: {question}"""
)

decompose = DECOMPOSE_PROMPT | llm | StrOutputParser() | parse_queries

complex_q = (
    "For a production deployment I need: uptime above 99.9 percent, "
    "encryption at rest, and GDPR compliance. Which Datanous Insight tier fits?"
)
sub_questions = decompose.invoke({"question": complex_q})
print(f"Complex question:\n  {complex_q}\n")
print("Sub-questions:")
for i, q in enumerate(sub_questions, 1):
    print(f"  {i}. {q}")


Complex question:
  For a production deployment I need: uptime above 99.9 percent, encryption at rest, and GDPR compliance. Which Datanous Insight tier fits?

Sub-questions:
  1. 1. What are the uptime guarantees for each Datanous Insight tier?
  2. 2. Does each Datanous Insight tier offer encryption at rest?
  3. 3. Which Datanous Insight tiers are compliant with GDPR?


Τώρα απαντάμε κάθε μία ξεχωριστά και συνθέτουμε:

In [8]:
SUB_ANSWER_PROMPT = ChatPromptTemplate.from_template(
    """Answer the question briefly based on the context below.

Context:
{context}

Question: {question}

Brief answer:"""
)

sub_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | SUB_ANSWER_PROMPT | llm | StrOutputParser()
)

qa_pairs = []
for sq in sub_questions:
    answer = sub_chain.invoke(sq)
    qa_pairs.append((sq, answer))
    print(f"Sub-Q: {sq}")
    print(f"Sub-A: {answer}\n")


Sub-Q: 1. What are the uptime guarantees for each Datanous Insight tier?
Sub-A: Datanous Insight Enterprise offers a 99.9 percent uptime SLA. The uptime guarantees for the Professional and Starter tiers are not specified in the context provided.

Sub-Q: 2. Does each Datanous Insight tier offer encryption at rest?
Sub-A: Yes, each Datanous Insight tier offers encryption at rest with AES-256.

Sub-Q: 3. Which Datanous Insight tiers are compliant with GDPR?
Sub-A: Both Datanous Insight Professional and Datanous Insight Enterprise tiers are compliant with GDPR.



In [9]:
# Synthesise a final answer from the sub-question answers
SYNTH_PROMPT = ChatPromptTemplate.from_template(
    """You have the following Q&A pairs that answer parts of a complex question.
Synthesise a single coherent final answer.

Q&A pairs:
{qa_pairs}

Original complex question: {question}

Final answer:"""
)

qa_text = "\n\n".join(f"Q: {q}\nA: {a}" for q, a in qa_pairs)
final   = (SYNTH_PROMPT | llm | StrOutputParser()).invoke(
    {"qa_pairs": qa_text, "question": complex_q}
)
print("Final synthesised answer:")
print(final)


Final synthesised answer:
For a production deployment requiring uptime above 99.9 percent, encryption at rest, and GDPR compliance, the Datanous Insight Enterprise tier is the only option that fits, as it offers a 99.9 percent uptime SLA, encryption at rest with AES-256, and is compliant with GDPR. The Professional tier also provides encryption at rest and GDPR compliance, but it does not meet the uptime requirement.


## 3.4b Decomposition — Recursive variant (IRCoT style)

Στο **3.4** είδαμε το *answer-individually* pattern: κάθε υπο-ερώτηση απαντιέται ανεξάρτητα και μετά συντίθεται η τελική απάντηση.

Υπάρχει όμως και μια **recursive** παραλλαγή (εμπνευσμένη από το IRCoT paper, arxiv 2212.10509):

> Η απάντηση κάθε υπο-ερώτησης γίνεται **context** για την επόμενη.
> Κάθε νέα ερώτηση «βλέπει» τις προηγούμενες απαντήσεις ⟹ cumulative reasoning.

<img src="images/03_IRCoT.png" height="700px" style="border-radius:10px;margin:12px 0;"/>

**Πότε να χρησιμοποιείς recursive:**
- Ερωτήσεις multi-hop reasoning (πχ «Ποιο tier είναι φθηνότερο από το Pulsar αλλά ακριβότερο από το Nova, και ποιο SLA έχει;»)
- Όταν κάθε υπο-ερώτηση εξαρτάται από την προηγούμενη απάντηση

**Πότε individual:**
- Ανεξάρτητες υπο-ερωτήσεις χωρίς dependency
- Θέλεις parallelism (batch call)


In [10]:
# Recursive (IRCoT-style) decomposition: each sub-answer informs the next sub-question
#
# KEY FIX: the retriever expects a plain string, not the full input dict.
# Use (lambda x: x["question"]) to extract the string before passing to retriever.

RECURSIVE_PROMPT = ChatPromptTemplate.from_template(
    """Answer the new question using the context AND any previous Q&A pairs provided.

Previous Q&A:
{previous_qa}

New context (from retrieval):
{context}

New question: {question}

Brief answer:"""
)

recursive_sub_chain = (
    {
        # Extract question string FIRST, then pipe to retriever
        "context":     (lambda x: x["question"]) | retriever | format_docs,
        "question":    lambda x: x["question"],
        "previous_qa": lambda x: x.get("previous_qa", "None"),
    }
    | RECURSIVE_PROMPT | llm | StrOutputParser()
)

def recursive_decompose_and_answer(complex_q: str) -> str:
    """
    1. Decompose into sub-questions.
    2. Answer Q1. Use A1 as context for Q2. And so on.
    3. Synthesise final answer.
    """
    sub_qs = decompose.invoke({"question": complex_q})
    print(f"Sub-questions: {len(sub_qs)}")
    qa_acc = []

    for i, sq in enumerate(sub_qs, 1):
        prev = "\n".join(f"Q{j}: {q}\nA{j}: {a}" for j, (q, a) in enumerate(qa_acc, 1))
        ans  = recursive_sub_chain.invoke({"question": sq, "previous_qa": prev or "None"})
        qa_acc.append((sq, ans))
        print(f"  Step {i} - Q: {sq}")
        print(f"          A: {ans}\n")

    qa_text = "\n\n".join(f"Q: {q}\nA: {a}" for q, a in qa_acc)
    final   = (SYNTH_PROMPT | llm | StrOutputParser()).invoke(
        {"qa_pairs": qa_text, "question": complex_q}
    )
    return final

recursive_q = (
    "I need a Datanous.ai solution that supports GDPR compliance, "
    "handles over 50,000 document pages, and provides an uptime SLA."
)
result = recursive_decompose_and_answer(recursive_q)
print("Final answer (recursive):")
print(result)


Sub-questions: 3
  Step 1 - Q: 1. What features does Datanous.ai offer to support GDPR compliance?
          A: Datanous.ai supports GDPR compliance by providing a Data Processing Agreement for Professional and Enterprise clients, ensuring that data handling practices align with GDPR requirements. Additionally, all deployments encrypt data in transit with TLS 1.3 and at rest with AES-256, further enhancing data protection and compliance.

  Step 2 - Q: 2. Can Datanous.ai handle more than 50,000 document pages efficiently?
          A: Yes, Datanous.ai can handle more than 50,000 document pages efficiently with its Datanous Insight Enterprise plan, which offers unlimited pages and a dedicated vector store.

  Step 3 - Q: 3. What uptime SLA does Datanous.ai provide for its services?
          A: Datanous.ai provides a 99.9 percent uptime SLA for its services.

Final answer (recursive):
Datanous.ai offers a comprehensive solution that supports GDPR compliance, efficiently handles over 50,

### Σύγκριση: Individual vs Recursive

| | Individual | Recursive (IRCoT) |
|---|---|---|
| **Parallelism** | ✅ Μπορείς batch | ❌ Sequential |
| **Multi-hop** | ⚠️ Χάνει dependencies | ✅ Εκμεταλλεύεται τις |
| **Cost** | $ (1 retrieval/sub-q) | $$ (1 retrieval + growing context/sub-q) |
| **Ιδανικό για** | Independent facts | Chain-of-thought reasoning |

> 📎 **Paper:** Interleaving Retrieval with Chain-of-Thought (IRCoT) — arxiv.org/abs/2212.10509


## 3.5 Step-Back — από συγκεκριμένο σε γενικό

Μερικές φορές μια **πολύ συγκεκριμένη** ερώτηση δεν matchάρει τίποτα, ενώ μια **πιο γενική** εκδοχή της θα έβρισκε τα σχετικά docs.

Παράδειγμα:
* Συγκεκριμένη: «Πώς ρυθμίζω auto-scaling για deploy.yaml με 4 replicas;»
* Step-back: «Πώς λειτουργεί το auto-scaling στο Cosmic Compute;»

Στο Step-Back retrieval κάνουμε **και** την αρχική **και** τη γενικευμένη ερώτηση, και χρησιμοποιούμε και τις δύο για context.

In [11]:
STEP_BACK_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert at query reformulation. "
     "Given a very specific question, produce a broader, more general version "
     "(a step-back question) whose answer will help answer the original."),
    ("human", "Original: How do I set a 30-day document retention policy in Datanous Insight Enterprise?"),
    ("ai",    "Step-back: How does Datanous Insight handle document lifecycle and retention management?"),
    ("human", "Original: {question}"),
])

step_back = STEP_BACK_PROMPT | llm | StrOutputParser()

specific_q = "How do I configure tenant isolation for a multi-tenant Datanous Insight Professional deployment?"
general_q  = step_back.invoke({"question": specific_q})

print(f"Original  : {specific_q}")
print(f"Step-back : {general_q}")


Original  : How do I configure tenant isolation for a multi-tenant Datanous Insight Professional deployment?
Step-back : Step-back: What are the best practices for managing tenant isolation in multi-tenant software deployments?


In [12]:
# Full step-back chain: retrieve for both the specific and the general question,
# then combine both contexts to answer the original.

STEP_BACK_FINAL = ChatPromptTemplate.from_template(
    """You have two sets of context:
1) From the specific question
2) From the broader step-back question

Combine both to answer the original question accurately.

Context (specific):
{specific_context}

Context (step-back):
{general_context}

Original question: {question}

Answer:"""
)

step_back_chain = (
    {
        "specific_context": (lambda x: x["question"]) | retriever | format_docs,
        "general_context":  (lambda x: x["question"]) | step_back | retriever | format_docs,
        "question":         lambda x: x["question"],
    }
    | STEP_BACK_FINAL | llm | StrOutputParser()
)

answer = step_back_chain.invoke({"question": specific_q})
print("Step-back RAG answer:")
print(answer)


Step-back RAG answer:
To configure tenant isolation for a multi-tenant Datanous Insight Professional deployment, you will need to utilize the built-in multi-tenant row-level access control feature. This feature allows you to manage access to data based on tenant identifiers, ensuring that each tenant can only access their own data.

Here are the steps to configure tenant isolation:

1. **Define Tenant Identifiers**: Assign a unique tenant_id to each tenant in your deployment. This identifier will be used to filter data and enforce isolation.

2. **Implement Row-Level Access Control**: Use the multi-tenant row-level access control feature to set up rules that restrict data access based on the tenant_id. This ensures that queries and data retrievals only return data relevant to the specific tenant making the request.

3. **Integrate Datanous Guard Checks**: Ensure that Datanous Guard is configured to enforce checks such as tenant isolation via tenant_id filtering. This will add an additi

## 3.6 HyDE — Hypothetical Document Embeddings

**Παρατήρηση:** Η ερώτηση και το doc έχουν συχνά **πολύ διαφορετικό «σχήμα»**:
* Ερώτηση: σύντομη, ερωτηματική
* Document: εκτεταμένο, καταφατικό

Στο **HyDE** ζητάμε από το LLM να γράψει **μια υποθετική απάντηση/document** στην ερώτηση. Μετά κάνουμε embedding **αυτή την υποθετική απάντηση** (όχι την ερώτηση) και ψάχνουμε.

Η ιδέα είναι ότι ένα **fake document** matchάρει καλύτερα με τα **real documents** από όσο matchάρει η ερώτηση.

<img src="images/03_HyDE.png" height="50%" width="70%" style="border-radius:10px;margin:12px 0;"/>


In [13]:
HYDE_PROMPT = ChatPromptTemplate.from_template(
    """Write a short, hypothetical technical passage (1-2 sentences) that would
answer the question below. You may invent plausible details.

Question: {question}

Hypothetical passage:"""
)

generate_hyde = HYDE_PROMPT | llm | StrOutputParser()

question   = "How does Datanous Search handle queries over very large document corpora?"
hypothetical = generate_hyde.invoke({"question": question})

print(f"Question          : {question}")
print(f"Hypothetical doc  : {hypothetical}")


Question          : How does Datanous Search handle queries over very large document corpora?
Hypothetical doc  : Datanous Search employs a distributed indexing architecture that segments large document corpora into manageable shards, allowing for parallel processing of queries across multiple nodes. By utilizing advanced algorithms for relevance ranking and caching frequently accessed data, the system ensures rapid retrieval times even when handling billions of documents.


In [14]:
# Compare: what does naive retrieval find vs HyDE retrieval?
naive_docs = retriever.invoke(question)
hyde_docs  = retriever.invoke(hypothetical)

print("Naive retrieval (query embedded directly):")
for d in naive_docs:
    print(f"  - {d.page_content[:90]}")

print("\nHyDE retrieval (hypothetical document embedded):")
for d in hyde_docs:
    print(f"  - {d.page_content[:90]}")


Naive retrieval (query embedded directly):
  - Datanous Search is a hybrid retrieval API combining BM25 and dense semantic embeddings via
  - Datanous Embed supports text-embedding-3-small (1536 dims) and text-embedding-3-large (307
  - Datanous Insight is a RAG-powered knowledge management platform that indexes enterprise do

HyDE retrieval (hypothetical document embedded):
  - Datanous Search is a hybrid retrieval API combining BM25 and dense semantic embeddings via
  - Datanous Insight is a RAG-powered knowledge management platform that indexes enterprise do
  - Datanous.ai serves clients in banking, legal, healthcare, and media across Europe. A law f


In [15]:
hyde_chain = (
    {
        "context":  generate_hyde | retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | FINAL_PROMPT | llm | StrOutputParser()
)

answer = hyde_chain.invoke(question)
print("HyDE RAG answer:")
print(answer)


HyDE RAG answer:
Datanous Search handles queries over very large document corpora by using a hybrid retrieval API that combines BM25 and dense semantic embeddings via Reciprocal Rank Fusion. Additionally, a cross-encoder reranker selects the final top-k results, ensuring efficient retrieval with latency under 200 ms at the 95th percentile over one million chunks.


# One more example... :D

In [23]:
from uuid import uuid4

from IPython.display import display, Markdown

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma


# ============================================================
# 1. LLM + Embeddings
# ============================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# ============================================================
# 2. Synthetic music-streaming fraud-detection corpus
# ============================================================
# Σχεδιασμένο ώστε D1, D2, D4, D5 να μοιράζονται επιφανειακές λέξεις
# με την ερώτηση ("fake streams", "detect", "flagged", "suspicious"),
# ενώ το πραγματικά σωστό D3 να χρησιμοποιεί τεχνικό ML λεξιλόγιο.

docs = [
    Document(
        page_content="""
        Η Datanous δημοσιεύει μια δημόσια σελίδα κέντρου βοήθειας που εξηγεί
        στους ακροατές και στους καλλιτέχνες τι είναι τα ψεύτικα streams,
        γιατί βλάπτουν το μουσικό οικοσύστημα και πώς οι χρήστες μπορούν να
        εντοπίζουν και να αναφέρουν ύποπτες λίστες αναπαραγωγής ή λογαριασμούς.
        Η σελίδα απαντά σε συχνές ερωτήσεις σχετικά με την αυθεντικότητα των
        streams, τις ψεύτικες αναπαραγωγές, τους λογαριασμούς bot και τη διαφορά
        ανάμεσα στην οργανική και την τεχνητή ακουστική συμπεριφορά. Το έγγραφο
        είναι καθαρά εκπαιδευτικό και δεν περιγράφει πώς λειτουργεί εσωτερικά η
        διαδικασία ανίχνευσης.
        """,
        metadata={
            "id": "D1",
            "title": "Σελίδα κέντρου βοήθειας σχετικά με ψεύτικες ροές"
        }
    ),

    Document(
        page_content="""
        Ο πίνακας ελέγχου καλλιτεχνών μέσα στη Datanous επιτρέπει σε μουσικούς
        και δισκογραφικές εταιρείες να εξετάζουν τον αριθμό των streams τους,
        να βλέπουν ποια κομμάτια επισημάνθηκαν ως ύποπτα ή ψεύτικα, να φιλτράρουν
        ανά χώρα, συσκευή ή πηγή λίστας αναπαραγωγής και να ελέγχουν τις
        καθημερινές αφαιρέσεις ψεύτικων streams. Ο πίνακας εμφανίζει
        επισημασμένους λογαριασμούς, αφαιρεμένες αναπαραγωγές και την κατάσταση
        ένστασης για κάθε εντοπισμένη περίπτωση. Το έγγραφο εστιάζει στη
        συμπεριφορά του UI και στις προβολές αναφορών, όχι στο υποκείμενο
        μοντέλο ανίχνευσης.
        """,
        metadata={
            "id": "D2",
            "title": "Πίνακας ελέγχου καλλιτέχνη για αναλυτικά στοιχεία ροής"
        }
    ),

    Document(
        page_content="""
        Η υπηρεσία βαθμολόγησης ακεραιότητας streams λαμβάνει ακατέργαστα
        συμβάντα αναπαραγωγής και κατασκευάζει κυλιόμενους γράφους συνεδριών
        διάρκειας 30 λεπτών, συνδέοντας λογαριασμούς ακροατών, αποτυπώματα
        συσκευών, υποδίκτυα IP και ακολουθίες κομματιών. Εξάγει χαρακτηριστικά
        όπως εντροπία διαστημάτων μεταξύ αναπαραγωγών, ανωμαλίες στο ποσοστό
        παράλειψης, σπανιότητα συνύπαρξης σε λίστες αναπαραγωγής, φήμη ASN,
        ταχύτητα εναλλαγής συσκευών και αναλογίες ηλικίας λογαριασμού προς όγκο
        ακροάσεων. Ένας βαθμονομημένος ταξινομητής gradient-boosted εκπέμπει
        πιθανότητα απάτης ανά συνεδρία, ενώ ένα επόμενο επίπεδο ανίχνευσης
        κοινοτήτων σε γράφο συγκεντρώνει συντονισμένες συστάδες. Οι
        αναπαραγωγές ακυρώνονται μόνο όταν η βαθμολογία της συστάδας υπερβαίνει
        ένα κατώφλι ειδικό για κάθε αγορά και παραμένει πάνω από αυτό σε
        διαδοχικά παράθυρα αξιολόγησης.
        """,
        metadata={
            "id": "D3",
            "title": "Αγωγός βαθμολόγησης ακεραιότητας ροής"
        }
    ),

    Document(
        page_content="""
        Η Datanous παρέχει μια ροή εργασίας συντονισμού, όπου οι αναλυτές
        εμπιστοσύνης και ασφάλειας διαχειρίζονται επισημασμένους λογαριασμούς,
        εξετάζουν αναφερόμενες λίστες αναπαραγωγής, προσθέτουν σημειώσεις
        έρευνας, χαρακτηρίζουν ύποπτες περιπτώσεις ως επιβεβαιωμένες ή
        απορριφθείσες και ενεργοποιούν αναστολές λογαριασμών για εντοπισμένη
        δραστηριότητα ψεύτικων streams. Η ροή εργασίας αποθηκεύει συμβάντα
        ελέγχου που δείχνουν ποιος αναλυτής χειρίστηκε κάθε αναφορά ψεύτικων
        streams και πότε ανοίχτηκε μια ένσταση. Το έγγραφο περιγράφει τη
        διαχείριση υποθέσεων και την παρακολούθηση αξιολόγησης, όχι την
        προγνωστική μοντελοποίηση.
        """,
        metadata={
            "id": "D4",
            "title": "Ροή εργασίας εποπτείας εμπιστοσύνης και ασφάλειας"
        }
    ),

    Document(
        page_content="""
        Η μονάδα αναφορών δικαιωμάτων συνοψίζει μηνιαίες μετρικές της πλατφόρμας,
        όπως συνολικά streams, αφαιρεθέντα ψεύτικα streams, προσαρμογές πληρωμών
        ανά δισκογραφική εταιρεία, μέση διάρκεια συνεδρίας ακροατή, πλήθος
        ύποπτων δραστηριοτήτων ανά αγορά και κατανομή εσόδων ανά χώρα. Οι
        αναφορές εξάγονται προς τους δικαιούχους για οικονομική συμφωνία.
        """,
        metadata={
            "id": "D5",
            "title": "Αναφορά δικαιωμάτων εκμετάλλευσης και μετρήσεις πληρωμήςs"
        }
    ),
]


# ============================================================
# 3. Vector store
# ============================================================

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name=f"hyde_streaming_demo_{uuid4().hex}",
)


# ============================================================
# 4. Answer prompt
# ============================================================

ANSWER_PROMPT = ChatPromptTemplate.from_template(
    """
    Answer the question using ONLY the retrieved context.

    If the retrieved context is only about help pages, artist dashboards,
    moderation workflow, royalty reporting, or UI behavior, answer exactly:
    "Το ανακτηθέν πλαίσιο δεν επαρκεί για να εξηγήσει τον μηχανισμό ανίχνευσης."

    Question:
    {question}

    Retrieved context:
    {context}

    Answer in Greek:
    """
)

answer_chain = ANSWER_PROMPT | llm | StrOutputParser()


# ============================================================
# 5. Helpers
# ============================================================

def format_context(docs_list):
    return "\n\n".join(
        f"id={doc.metadata['id']} | title={doc.metadata['title']}\n"
        f"{doc.page_content.strip()}"
        for doc in docs_list
    )


def retrieval_table(results):
    rows = [
        "| Rank | Document ID | Title | Distance |",
        "|---:|---|---|---:|"
    ]

    for rank, (doc, distance) in enumerate(results, start=1):
        rows.append(
            f"| {rank} | {doc.metadata['id']} | "
            f"{doc.metadata['title']} | {distance:.4f} |"
        )

    return "\n".join(rows)


def render_markdown_report(
    question,
    hypothetical_doc,
    baseline_results,
    hyde_results,
    baseline_answer,
    hyde_answer
):
    baseline_top_id = baseline_results[0][0].metadata["id"]
    hyde_top_id = hyde_results[0][0].metadata["id"]

    if baseline_top_id != "D3" and hyde_top_id == "D3":
        verdict = "**Clear HyDE win**"
    elif baseline_top_id == "D3" and hyde_top_id == "D3":
        verdict = "**Both methods found the correct document. The corpus is not hard enough.**"
    elif baseline_top_id != "D3" and hyde_top_id != "D3":
        verdict = "**HyDE did not recover the correct document. The HyDE passage needs to be more technical.**"
    else:
        verdict = "**Unexpected result: baseline found the correct document but HyDE did not.**"

    markdown = f"""
# HyDE vs Baseline RAG — Music Streaming Fraud Detection Demo

## Question

> {question}

---

## Hypothetical Document Used by HyDE

> {hypothetical_doc.strip()}

---

## Baseline Retrieval  
Query used: original user question

{retrieval_table(baseline_results)}

---

## HyDE Retrieval  
Query used: hypothetical technical document

{retrieval_table(hyde_results)}

---

## Baseline Answer  
Using only top-1 retrieved document: **{baseline_top_id}**

> {baseline_answer}

---

## HyDE Answer  
Using only top-1 retrieved document: **{hyde_top_id}**

> {hyde_answer}

---

## Summary

| Method | Top-1 Document | Correct Document? |
|---|---|---|
| Baseline | {baseline_top_id} | {"Yes" if baseline_top_id == "D3" else "No"} |
| HyDE | {hyde_top_id} | {"Yes" if hyde_top_id == "D3" else "No"} |

**Result:** {verdict}
"""

    display(Markdown(markdown))


# ============================================================
# 6. Question
# ============================================================
# Καθημερινή διατύπωση από τη μεριά του χρήστη.
# Μοιράζεται λέξεις με D1, D2, D4, D5 ("fake streams", "detect"),
# ΟΧΙ με το D3 (που μιλάει για "stream integrity", "fraud probability",
# "session graphs", "coordinated clusters" κλπ).

question = "How does Datanous detect fake streams on the platform?"


# ============================================================
# 7. Baseline retrieval
# ============================================================

baseline_results = vectorstore.similarity_search_with_score(
    query=question,
    k=3
)

baseline_top1_doc = [baseline_results[0][0]]

baseline_answer = answer_chain.invoke({
    "question": question,
    "context": format_context(baseline_top1_doc)
})


# ============================================================
# 8. HyDE retrieval
# ============================================================
# Cached HyDE passage για ντετερμινιστική επίδειξη στο μάθημα.
# Σε production θα παραγόταν δυναμικά από έναν generator-LLM.

hypothetical_doc = """
Το Datanous ανιχνεύει τεχνητή δραστηριότητα ροής μοντελοποιώντας κυλιόμενα γραφήματα 30λεπτων
συνεδριών ακρόασης που συνδέουν λογαριασμούς ακροατών, δακτυλικά αποτυπώματα συσκευών,
υποδίκτυα IP και ακολουθίες κομματιών. Η υπηρεσία βαθμολόγησης ακεραιότητας ροής εξάγει
χαρακτηριστικά όπως η εντροπία διαστήματος αλληλεπίδρασης, οι ανωμαλίες ρυθμού παράλειψης, η σπανιότητα συνύπαρξης λίστας αναπαραγωγής, η φήμη ASN, η ταχύτητα εναλλαγής συσκευής και οι
αναλογίες ηλικίας λογαριασμού έναντι όγκου ακρόασης. Ένας βαθμονομημένος ταξινομητής με ενίσχυση κλίσης
παράγει μια πιθανότητα απάτης ανά συνεδρία, ενώ ένα επίπεδο ανίχνευσης κοινότητας γραφήματος
συγκεντρώνει συντονισμένα συμπλέγματα bot. Οι αναπαραγωγές ακυρώνονται μόνο όταν η
βαθμολογία συμπλέγματος υπερβαίνει ένα όριο που αφορά την αγορά σε διαδοχικά
παράθυρα αξιολόγησης.
"""

hyde_results = vectorstore.similarity_search_with_score(
    query=hypothetical_doc,
    k=3
)

hyde_top1_doc = [hyde_results[0][0]]

hyde_answer = answer_chain.invoke({
    "question": question,
    "context": format_context(hyde_top1_doc)
})


# ============================================================
# 9. Markdown output
# ============================================================

render_markdown_report(
    question=question,
    hypothetical_doc=hypothetical_doc,
    baseline_results=baseline_results,
    hyde_results=hyde_results,
    baseline_answer=baseline_answer,
    hyde_answer=hyde_answer
)


# HyDE vs Baseline RAG — Music Streaming Fraud Detection Demo

## Question

> How does Datanous detect fake streams on the platform?

---

## Hypothetical Document Used by HyDE

> Το Datanous ανιχνεύει τεχνητή δραστηριότητα ροής μοντελοποιώντας κυλιόμενα γραφήματα 30λεπτων
συνεδριών ακρόασης που συνδέουν λογαριασμούς ακροατών, δακτυλικά αποτυπώματα συσκευών,
υποδίκτυα IP και ακολουθίες κομματιών. Η υπηρεσία βαθμολόγησης ακεραιότητας ροής εξάγει
χαρακτηριστικά όπως η εντροπία διαστήματος αλληλεπίδρασης, οι ανωμαλίες ρυθμού παράλειψης, η σπανιότητα συνύπαρξης λίστας αναπαραγωγής, η φήμη ASN, η ταχύτητα εναλλαγής συσκευής και οι
αναλογίες ηλικίας λογαριασμού έναντι όγκου ακρόασης. Ένας βαθμονομημένος ταξινομητής με ενίσχυση κλίσης
παράγει μια πιθανότητα απάτης ανά συνεδρία, ενώ ένα επίπεδο ανίχνευσης κοινότητας γραφήματος
συγκεντρώνει συντονισμένα συμπλέγματα bot. Οι αναπαραγωγές ακυρώνονται μόνο όταν η
βαθμολογία συμπλέγματος υπερβαίνει ένα όριο που αφορά την αγορά σε διαδοχικά
παράθυρα αξιολόγησης.

---

## Baseline Retrieval  
Query used: original user question

| Rank | Document ID | Title | Distance |
|---:|---|---|---:|
| 1 | D1 | Σελίδα κέντρου βοήθειας σχετικά με ψεύτικες ροές | 1.0336 |
| 2 | D2 | Πίνακας ελέγχου καλλιτέχνη για αναλυτικά στοιχεία ροής | 1.0589 |
| 3 | D4 | Ροή εργασίας εποπτείας εμπιστοσύνης και ασφάλειας | 1.1297 |

---

## HyDE Retrieval  
Query used: hypothetical technical document

| Rank | Document ID | Title | Distance |
|---:|---|---|---:|
| 1 | D3 | Αγωγός βαθμολόγησης ακεραιότητας ροής | 0.5443 |
| 2 | D4 | Ροή εργασίας εποπτείας εμπιστοσύνης και ασφάλειας | 0.5647 |
| 3 | D1 | Σελίδα κέντρου βοήθειας σχετικά με ψεύτικες ροές | 0.9959 |

---

## Baseline Answer  
Using only top-1 retrieved document: **D1**

> Το ανακτηθέν πλαίσιο δεν επαρκεί για να εξηγήσει τον μηχανισμό ανίχνευσης.

---

## HyDE Answer  
Using only top-1 retrieved document: **D3**

> Η Datanous ανιχνεύει ψεύτικες ροές στην πλατφόρμα μέσω μιας υπηρεσίας βαθμολόγησης ακεραιότητας ροής που αναλύει ακατέργαστα συμβάντα αναπαραγωγής. Δημιουργεί κυλιόμενους γράφους συνεδριών, συνδέοντας λογαριασμούς ακροατών, αποτυπώματα συσκευών, υποδίκτυα IP και ακολουθίες κομματιών. Εξάγει χαρακτηριστικά όπως εντροπία διαστημάτων μεταξύ αναπαραγωγών, ανωμαλίες στο ποσοστό παράλειψης και άλλες παραμέτρους. Ένας βαθμονομημένος ταξινομητής gradient-boosted υπολογίζει την πιθανότητα απάτης ανά συνεδρία, ενώ ανιχνεύει και συντονισμένες συστάδες μέσω ενός επιπέδου ανίχνευσης κοινοτήτων. Οι αναπαραγωγές ακυρώνονται όταν η βαθμολογία της συστάδας υπερβαίνει ένα συγκεκριμένο κατώφλι.

---

## Summary

| Method | Top-1 Document | Correct Document? |
|---|---|---|
| Baseline | D1 | No |
| HyDE | D3 | Yes |

**Result:** **Clear HyDE win**


## 3.7 Πότε να χρησιμοποιήσεις τι

| Τεχνική | Καλύτερη όταν | Cost |
|---|---|---|
| **Naive RAG** | Vocabulary του query ≈ vocabulary του corpus | $ (1 retrieval) |
| **Multi-Query** | Διαφορετικά συνώνυμα παίζουν ρόλο | $$ (1 LLM + 4 retrievals) |
| **RAG-Fusion** | Χρειάζεσαι robust ranking | $$ (1 LLM + 4 retrievals + RRF) |
| **Decomposition** | Σύνθετες multi-faceted ερωτήσεις | $$$ (πολλά LLM calls) |
| **Step-Back** | Πολύ συγκεκριμένες ερωτήσεις σε generic corpus | $$ (2 retrievals) |
| **HyDE** | Mismatch query↔doc style (πχ tech support FAQs) | $$ (1 LLM + 1 retrieval) |

**Pro tip:** Δεν χρειάζεται να επιλέξεις μία. Σε production συνδυάζονται. Για παράδειγμα: HyDE + RAG-Fusion + reranker (notebook 06).

## 3.8 Άσκηση

**Άσκηση:** Φτιάξε μια συνάρτηση `compare_retrievers(question)` που εμφανίζει side-by-side τα top-3 results από:

1. Naive retrieval
2. Multi-Query (union)
3. RAG-Fusion (RRF)
4. HyDE

Για κάθε μέθοδο εκτύπωσε σύντομη preview των docs.

In [ ]:
# Exercise: Build a compare_strategies() function that runs four retrieval strategies
# on the same question and prints the top-3 results for each:
#   1. Naive        — retriever.invoke(question)
#   2. Multi-Query  — multi_query_retrieval(question)
#   3. HyDE         — retriever.invoke(generate_hyde.invoke(question))
#   4. Step-back    — retriever.invoke(step_back.invoke({"question": question}))
#
# Test with: compare_strategies("What data protection guarantees does Datanous.ai offer?")

# Your solution here:


In [ ]:
def compare_strategies(question: str, top_k: int = 3):
    """Compare four query transformation strategies side by side."""
    print(f"Question: {question}")
    print("=" * 80)

    # 1. Naive
    print("\n[1] NAIVE")
    for d in retriever.invoke(question)[:top_k]:
        print(f"  - {d.page_content[:80]}")

    # 2. Multi-Query (RRF)
    print("\n[2] MULTI-QUERY (RRF)")
    for doc, score in multi_query_retrieval(question)[:top_k]:
        print(f"  - [{score:.4f}] {doc.page_content[:75]}")

    # 3. HyDE
    print("\n[3] HyDE")
    hyp = generate_hyde.invoke({"question": question})
    for d in retriever.invoke(hyp)[:top_k]:
        print(f"  - {d.page_content[:80]}")

    # 4. Step-back
    print("\n[4] STEP-BACK")
    general = step_back.invoke({"question": question})
    for d in retriever.invoke(general)[:top_k]:
        print(f"  - {d.page_content[:80]}")

compare_strategies("What data protection guarantees does Datanous.ai offer?")


## 📋 Συμπεράσματα

| # | Έννοια | Συνοπτικά |
|---|---|---|
| 1 | Πρόβλημα | Vocabulary mismatch query ↔ docs |
| 2 | Multi-Query | Παράγουμε 4 παραλλαγές → union των retrievals |
| 3 | RAG-Fusion | Multi-Query + RRF για smart ranking |
| 4 | RRF formula | $\text{score}(d) = \sum_q \frac{1}{k + \text{rank}_q(d)}$ |
| 5 | RRF k constant | Συνήθως 60 — αμβλύνει τα τοπ ranks |
| 6 | Decomposition | Σπάμε σύνθετη ερώτηση σε υπο-ερωτήσεις, συνθέτουμε |
| 7 | Step-Back | Από συγκεκριμένο σε γενικό → καλύτερο coverage |
| 8 | HyDE | Embed υποθετικού doc αντί της ερώτησης |
| 9 | Cost trade-off | Κάθε τεχνική = +LLM calls + retrievals |
| 10 | Composability | Συνδυάζονται μεταξύ τους και με reranking |
| 11 | Επόμενο βήμα | Notebook 04 — Routing & Query Construction |